In [1]:
# Import necessary libraries
import pandas as pd
import datetime as dt

In [ ]:
# Import cpi_month.xlsx from imf.org, extracted sheet "cpi_month"
cpi_data = pd.read_excel("cpi_month.xlsx", sheet_name="cpi_month")
cpi_data.head()

,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,INDEX_TYPE,COICOP_1999,TYPE_OF_TRANSFORMATION,2002-M01,2002-M02,2002-M03,...,2024-M12,2025-M01,2025-M02,2025-M03,2025-M04,2025-M05,2025-M06,2025-M07,2025-M08,2025-M09
0,IMF.STA:CPI(5.0.0),IND.CPI.CP03.IX.M,OBS_VALUE,India,Consumer price index (CPI),Clothing and footwear,Index,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IMF.STA:CPI(5.0.0),IND.CPI.CP02.IX.M,OBS_VALUE,India,Consumer price index (CPI),"Alcoholic beverages, tobacco and narcotics",Index,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IMF.STA:CPI(5.0.0),IND.CPI.CP10.IX.M,OBS_VALUE,India,Consumer price index (CPI),Education,Index,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IMF.STA:CPI(5.0.0),IND.CPI.CP08.IX.M,OBS_VALUE,India,Consumer price index (CPI),Communication,Index,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IMF.STA:CPI(5.0.0),IND.CPI.CP06.IX.M,OBS_VALUE,India,Consumer price index (CPI),Health,Index,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Filter for standard reference period and drop unnecessary columns
cpi_data = cpi_data[
    cpi_data["TYPE_OF_TRANSFORMATION"].isin([
        "Standard reference period (2010=100), Index",
        "Standard reference period (2010=100), Period average, Year-over-year (YOY) percent change"
    ])
]

cpi_data = cpi_data.drop(columns=["DATASET", "SERIES_CODE", "OBS_MEASURE", "INDEX_TYPE", "COICOP_1999", "TYPE_OF_TRANSFORMATION"])

In [4]:
# Remove index/column axis names
cpi_data = cpi_data.set_index("COUNTRY").T.reset_index()
cpi_data.rename(columns={"index": "Year_Month"}, inplace=True)
cpi_data.columns.name = None
cpi_data.columns = ["Year_Month", "CPI", "YoY_Inflation"]
print(cpi_data)


    Year_Month         CPI  YoY_Inflation
0     2002-M01   57.541961       4.943862
1     2002-M02   57.418721       5.191865
2     2002-M03   57.665144       5.168521
3     2002-M04   57.788385       4.687518
4     2002-M05   58.158048       4.656368
..         ...         ...            ...
280   2025-M05  230.127125       2.823655
281   2025-M06  231.557967       2.103049
282   2025-M07  233.823467       1.606218
283   2025-M08  234.896599       2.072539
284   2025-M09  235.135073       1.544799

[285 rows x 3 columns]


In [5]:
# Convert "Year_Month" to a proper datetime format
cpi_data["Year_Month"] = (cpi_data["Year_Month"].astype(str).str
                          .replace(r'[-\s]*M', '-', regex=True) + '-01')
cpi_data["Year_Month"] = pd.to_datetime(cpi_data["Year_Month"], format="%Y-%m-%d")
print(cpi_data.head(1))
print(cpi_data.tail(1))

  Year_Month        CPI  YoY_Inflation
0 2002-01-01  57.541961       4.943862
    Year_Month         CPI  YoY_Inflation
284 2025-09-01  235.135073       1.544799


In [6]:
# Check for missing values and data types
print("Total observations in each column =", len(cpi_data),"\n")
print(cpi_data.info())

Total observations in each column = 285 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285 entries, 0 to 284
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Year_Month     285 non-null    datetime64[ns]
 1   CPI            285 non-null    float64       
 2   YoY_Inflation  285 non-null    float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 6.8 KB
None


In [7]:
# Europe_Brent_Spot_Price_FOB.xlsx from eia.gov, transformed slightly in Excel to remove extra header rows
oil_data = pd.read_excel("Europe_Brent_Spot_Price_FOB.xlsx", sheet_name="Europe_Brent_Spot_Price_FOB")
print(oil_data.head())
print(oil_data.info())

       Month  Spot_Price
0 2025-10-01       64.54
1 2025-09-01       67.99
2 2025-08-01       67.87
3 2025-07-01       71.04
4 2025-06-01       71.44
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 462 entries, 0 to 461
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Month       462 non-null    datetime64[ns]
 1   Spot_Price  462 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 7.3 KB
None


In [8]:
# Convert "Month" to a proper datetime format
oil_data = oil_data.sort_values("Month")
oil_data = oil_data[(oil_data["Month"] >= dt.datetime(2002, 1, 1)) & (oil_data["Month"] <= dt.datetime(2025, 10, 1))]
print(oil_data.head(1))
print(oil_data.tail(1))

         Month  Spot_Price
285 2002-01-01       19.42
       Month  Spot_Price
0 2025-10-01       64.54


In [9]:
# Rename columns for easier access
oil_data.reset_index(drop=True, inplace=True)
oil_data.rename(columns={"Europe Brent Spot Price FOB Dollars per Barrel":"Spot_Price"}, inplace=True)

In [10]:
# Check for missing values and data types
print("Total observations in each column =", len(oil_data),"\n")
print(oil_data.info())

Total observations in each column = 286 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286 entries, 0 to 285
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Month       286 non-null    datetime64[ns]
 1   Spot_Price  286 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 4.6 KB
None


In [11]:
# Merge datasets on the date columns and extract Year and Month
merge_data = pd.merge(cpi_data, oil_data, left_on="Year_Month", right_on="Month", how="inner").drop(columns=["Month"])
merge_data["Year"] = merge_data["Year_Month"].dt.year
merge_data["Month"] = merge_data["Year_Month"].dt.month

cols = ["Year_Month", "Year", "Month", "Spot_Price", "CPI", "YoY_Inflation"]
merge_data = merge_data[cols]

In [12]:
print(merge_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285 entries, 0 to 284
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Year_Month     285 non-null    datetime64[ns]
 1   Year           285 non-null    int32         
 2   Month          285 non-null    int32         
 3   Spot_Price     285 non-null    float64       
 4   CPI            285 non-null    float64       
 5   YoY_Inflation  285 non-null    float64       
dtypes: datetime64[ns](1), float64(3), int32(2)
memory usage: 11.3 KB
None


In [13]:
# Export merged data to a new Excel file
merge_data.to_excel("merged_data.xlsx", index=False)